# Two domains, one model, no rehearsal

This notebook adapts **Qwen3-0.6B** to one domain, then to a second one, and measures at every
step what was learned and what was kept. Between the two domains nothing from the first is
stored, replayed or revisited: the only thing that carries the first domain forward is a
distribution of hidden states.

It is a demonstration of Layerwise Function Anchoring (LFA), and the point of it is the
**control**. Two of the runs below have the anchor switched off — same corpus, same optimizer
steps, same everything else — and the difference between the anchored and the unanchored
column is what the method bought. If the anchor did nothing, the two columns would agree.
Two further runs then check the controls themselves, by giving each its own best number
of epochs rather than the anchored run's.

## What happens

| | |
|---|---|
| 1 | Fetch the published `p(h)` artifact and look inside it |
| 2 | Download two public-domain books, one per domain |
| 3 | Make a workspace |
| 4 | **Stage 1** — adapt to domain A, and read both axes |
| 5 | **The control** — the same stage with the anchor off |
| 6 | `extend` — fold A into the model *and* into `p(h)` |
| 7 | **Stage 2** — adapt to domain B, anchored against the extended `p(h)` |
| 8 | **The control again** — stage 2 with the anchor off, and what happens to A |
| 8b | **A check on the controls themselves** — both, at their own best dose |
| 9 | Export a plain checkpoint and generate from it |

## What you need

* One CUDA card with about 12 GB free. Everything below runs on a single device.
* `pip install lfa-anchoring` (this notebook was run against version 0.1.0).
* Network, for the artifact, the model, the two books and WikiText-2.
* **No API keys.** Every number here is a perplexity computed on your own machine; nothing in
  this package calls out to a model service.

## What it costs, and what the real recipe costs

The settings in the next cell are **demo settings**, chosen so the whole notebook finishes in
well under an hour on one RTX 3090. They are not the recommended operating point.

| | this notebook | the shipped recipe (`lfa/recipes/qwen3-0.6b.yaml`) |
|---|---|---|
| text per domain | 0.97 MB and 0.48 MB | 6.6 MB |
| epochs | 5 | 15 |
| wall clock per training run | 2-4 min | ~1 h 48 m |
| training runs | 6 (two stages, two controls, two dose checks) | 1 per stage |
| whole notebook | 21 min | — |

Everything else — rank 32, λ = 100,000, μ = 0.05, the layer schedule, the optimizer — is the
shipped recipe untouched, because λ is coupled to the rank and to the artifact and porting it
is a different regularizer (`docs/concepts.md`). What this notebook changes is corpus size and
run length, and it says so at every number it prints.

One consequence worth stating up front: this is **one run per cell, one seed, no error bars**.
It shows a mechanism at a scale you can watch. It is not a measurement of the method, and the
paper's numbers are not reproduced here.

## A note on the outputs saved in this notebook

The outputs below were recorded on 2026-09-08, **before** a deliberate change to the corpus
loader, and a re-run will not reproduce their per-epoch numbers.

The per-epoch chunk offset used to be where each document *started*, so from the second epoch on
every document lost its first `offset` tokens. This notebook's corpora are exactly the shape that
hurts — each book is cut into ~3,500-character documents, a little over one 512-token chunk each —
and the damage is visible in the epoch lines below: stage 1 trains on 164,282 tokens in epoch 1 and
92,367 in epoch 3, and over the five epochs it sees about **83 %** of the token exposure it was
asked for (stage 2, the same offsets, the same 83 %).

The offset now rotates the chunk *boundaries* instead: the leading segment becomes a chunk of its
own, so every token is trained on in every epoch. On a re-run, expect the per-epoch token counts to
be flat rather than to swing, one extra chunk per multi-chunk document per epoch, an extra
`loader frame:` line naming the new frame, and every loss, perplexity and dose diagnosis to move
with them. What the notebook is *about* — the anchored column against the unanchored control —
does not depend on any of that, and both columns move together.

`rotate_offset=False` on `Workspace.train` restores the old stream for anyone reproducing these
exact numbers. See [`docs/verification.md`](../docs/verification.md).


In [1]:
from pathlib import Path

# --- demo scale (see the table above) ---------------------------------------------------------
EPOCHS = 5              # the shipped recipe's 15 over-trains a corpus this small; see cell "dose"
N_WINDOWS = 200         # WikiText-2 sliding windows for the general axis (200 x 512 tokens)
EXTEND_NEED = 20_000    # hidden states collected per site when folding domain A into p(h)
EXTEND_K = 4            # mixture components fitted per site for the new domain

DEVICE = "cuda:0"
ARTIFACT_ID = "qwen3-0.6b-gmm1543k-int8"
MODEL_ID = "Qwen/Qwen3-0.6B"

DEMO = Path("lfa_demo").resolve()      # everything this notebook creates lives here
DATA, ARTIFACTS, WS = DEMO / "data", DEMO / "artifacts", DEMO / "workspace"
DEMO.mkdir(exist_ok=True)
print(DEMO)

/tmp/lfa-nb-run/cwd/lfa_demo


In [2]:
import logging, os, sys, time, textwrap

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logging.getLogger("datasets").setLevel(logging.WARNING)

import torch
import lfa

NOTEBOOK_STARTED = time.time()
print(f"lfa          {lfa.__version__}")
print(f"torch        {torch.__version__}")
print(f"CUDA device  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "this notebook needs a CUDA card"

lfa          0.1.0
torch        2.10.0+cu128
CUDA device  NVIDIA GeForce RTX 3090


## 1. The artifact: what makes adaptation data-free

LFA's preservation signal does not come from text. It comes from a file, built **once per
model**, that records what the hidden states arriving at each anchored sub-module look like:
a mean, a PCA basis with its eigenvalues, and a small mixture fitted in that basis, per site.
That file is the artifact, and the training loop samples hidden states from it instead of
reading a corpus.

`fetch_artifact` downloads the published one (~108 MB) and checks its SHA-256 against the
registry before putting it in place, because a truncated or wrong artifact has no visible
failure mode — the shapes would still be right, the run would still train, and only the
preservation number would be off.

A file already at the destination is **verified rather than re-downloaded**, so if you already
have a copy, drop it in as `lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt` (or point the
`LFA_ARTIFACT` environment variable at it) and the same call checksums it in place. The run
whose outputs are saved in this notebook took that route — hence the `copied …` line below and
the 0-second fetch — so what you are reading is the verify-in-place branch, not the download.
It is the same function and the same checksum either way.

In [3]:
import shutil
from lfa.artifact.fetch import ARTIFACTS as REGISTRY, fetch_artifact, list_artifacts

for entry in list_artifacts():
    print(f"{entry['id']:<28} {entry['size_mb']:>4} MB   {entry['kind']}")

# An out-of-band copy is accepted at the destination name; fetch_artifact then verifies it
# against the registry checksum instead of downloading it again.
local_copy = os.environ.get("LFA_ARTIFACT")
target = ARTIFACTS / f"{ARTIFACT_ID}.pt"
if local_copy and not target.exists():
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(local_copy, target)
    print(f"\ncopied {local_copy} -> {target}; it will be checksum-verified, not trusted")

t0 = time.time()
artifact_path = fetch_artifact(ARTIFACT_ID, ARTIFACTS)
print(f"\n{artifact_path}  ({artifact_path.stat().st_size / 1e6:.1f} MB, {time.time()-t0:.0f}s)")

qwen3-0.6b-diagonal             1 MB   diagonal (budget floor)
qwen3-0.6b-gmm1543k-int8      108 MB   correlated-linear + GMM K=32 MLP (int8)

copied /tmp/lfa-nb-run/artifact/qwen3-0.6b-gmm1543k-int8.pt -> /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt; it will be checksum-verified, not trusted
Artifact qwen3-0.6b-gmm1543k-int8 already present at /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt (checksum verified)



/tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt  (112.8 MB, 0s)


Worth opening, because the claim that this is "not the old data in disguise" is checkable.
The artifact is a dictionary of per-site blocks and one metadata block, and the inventory of
what a block may hold is printed at the bottom of the next cell: tensors of statistics, integer
counts, floats, and one string per site naming the kind of mixture that was fitted. No token
ids, no documents, nothing sequence-shaped. The object is a *distribution over vectors*, and
the sites it covers are the inputs of the sub-modules being preserved.

In [4]:
from lfa.artifact.schema import load_artifact

params = load_artifact(artifact_path)      # int8 on disk, dequantized on load
meta = params["__meta__"]
sites = [k for k in params if k != "__meta__"]

print("__meta__:")
for key, value in meta.items():
    print(f"   {key:<18} {value if not isinstance(value, list) else value}")
print(f"\n{len(sites)} site blocks, e.g. {sites[:3]} ... {sites[-2:]}")

example = params["12_pre_mlp"]
print("\nwhat one site stores (layer 12, the input of the whole MLP):")
for key, value in example.items():
    shape = tuple(value.shape) if torch.is_tensor(value) else value
    print(f"   {key:<22} {shape}")

types = {type(v).__name__ for block in sites for v in params[block].values()}
print(f"\nvalue types across every site block: {sorted(types)}")

__meta__:
   model_id           Qwen/Qwen3-0.6B
   hidden_size        1024
   num_layers         28
   sites              ['pre_qkv', 'pre_o', 'pre_mlp', 'pre_lm_head']
   n_samples_total    1543040
   built_with         research code; meta block added by lfa-anchoring
   lfa_version        0.1.0

84 site blocks, e.g. ['0_pre_o', '0_pre_mlp', '1_pre_qkv'] ... ['27_pre_mlp', '28_pre_lm_head']

what one site stores (layer 12, the input of the whole MLP):
   mean                   (1024,)
   std                    (1024,)
   pca_components         (1024, 747)
   pca_eigenvalues        (747,)
   pca_n_components       747
   gmm_n_components       32
   gmm_weights            (32,)
   gmm_means              (32, 747)
   gmm_covariances        (32, 747)
   gmm_covariance_type    diag
   gmm_log_likelihood_p10 -799.129150390625
   gmm_log_likelihood_p50 -675.5284423828125
   gmm_log_likelihood_p90 -548.2757568359375
   gaussian_coverage_ratio 1.2710726009969582
   gaussian_nn_distance_mean 2

## 2. Two domains, from Project Gutenberg

Two books, both public domain, both a single plain-text download with no account and no
credentials. They are chosen to be *far apart* — different century, different purpose,
different vocabulary — so that "adapted to A" and "adapted to B" are visibly different states
of the same model, and so that neither one is WikiText-2, which is the general axis.

| | domain A | domain B |
|---|---|---|
| book | *On the Origin of Species* | *Domestic Cookery, Useful Receipts, and Hints to Young Housekeepers* |
| author, year | Charles Darwin, 1859 | Elizabeth E. Lea, 1853 |
| Gutenberg id | 1228 | 9101 |
| download | `https://www.gutenberg.org/cache/epub/1228/pg1228.txt` (0.97 MB) | `https://www.gutenberg.org/cache/epub/9101/pg9101.txt` (0.48 MB) |

**1.45 MB of text in total.** The cell below downloads them, strips the Gutenberg licence
header and footer, and accumulates paragraphs into documents of about 3,500 characters —
roughly 850 tokens, so most documents are one or two of the 512-token chunks the trainer reads.

Every eighth document goes to a `heldout/` directory and is **never trained on, at either
stage**. Every domain perplexity printed in this notebook is measured on those documents, so
all of them are comparable with each other. (The trainer separately keeps a tenth of the
*training* documents back for its own per-epoch curve; that is a different split, used only to
choose the number of epochs.)

In [5]:
import hashlib, re, urllib.request

BOOKS = {
    "darwin":  dict(gid=1228, title="On the Origin of Species (Charles Darwin, 1859)"),
    "cookery": dict(gid=9101, title="Domestic Cookery (Elizabeth E. Lea, 1853)"),
}
_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
_END = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)


def strip_gutenberg(raw: str) -> str:
    """Drop the licence header and footer, keeping the book."""
    if (m := _START.search(raw)):
        raw = raw[m.end():]
    if (m := _END.search(raw)):
        raw = raw[:m.start()]
    return raw.strip()


def to_documents(text: str, target_chars: int = 3500, min_chars: int = 1200) -> list[str]:
    """Accumulate blank-line-separated paragraphs into documents of about `target_chars`."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    documents, buffer, size = [], [], 0
    for paragraph in paragraphs:
        buffer.append(paragraph)
        size += len(paragraph) + 2
        if size >= target_chars:
            documents.append("\n\n".join(buffer))
            buffer, size = [], 0
    if size >= min_chars:
        documents.append("\n\n".join(buffer))
    return [d for d in documents if len(d) >= min_chars]


def prepare(name: str, hold_out_every: int = 8) -> dict:
    spec = BOOKS[name]
    raw_path = DATA / "raw" / f"pg{spec['gid']}.txt"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    if not raw_path.exists():
        url = f"https://www.gutenberg.org/cache/epub/{spec['gid']}/pg{spec['gid']}.txt"
        with urllib.request.urlopen(url, timeout=120) as response:
            raw_path.write_bytes(response.read())
    raw = raw_path.read_bytes()
    documents = to_documents(strip_gutenberg(raw.decode("utf-8", errors="replace")))

    train, heldout = DATA / name / "train", DATA / name / "heldout"
    for directory in (train, heldout):
        directory.mkdir(parents=True, exist_ok=True)
        for stale in directory.glob("*.txt"):
            stale.unlink()
    for i, document in enumerate(documents):
        destination = heldout if i % hold_out_every == hold_out_every - 1 else train
        (destination / f"{i:04d}.txt").write_text(document, encoding="utf-8")

    return dict(title=spec["title"], mb=len(raw) / 1e6,
                sha256=hashlib.sha256(raw).hexdigest()[:12],
                train=len(list(train.glob("*.txt"))), heldout=len(list(heldout.glob("*.txt"))))


corpora = {name: prepare(name) for name in BOOKS}
for name, info in corpora.items():
    print(f"{name:<9} {info['title']}\n"
          f"          {info['mb']:.2f} MB downloaded, sha256 {info['sha256']}…, "
          f"{info['train']} training documents + {info['heldout']} held out")

DARWIN_TRAIN, DARWIN_HELD = DATA / "darwin" / "train", DATA / "darwin" / "heldout"
COOKERY_TRAIN, COOKERY_HELD = DATA / "cookery" / "train", DATA / "cookery" / "heldout"

darwin    On the Origin of Species (Charles Darwin, 1859)
          0.97 MB downloaded, sha256 ededa9c0bf87…, 193 training documents + 27 held out
cookery   Domestic Cookery (Elizabeth E. Lea, 1853)
          0.48 MB downloaded, sha256 8ab8a0d20e54…, 105 training documents + 15 held out


A paragraph of each, so it is clear how different the two domains are:

In [6]:
for name, path in (("A / darwin", DARWIN_TRAIN), ("B / cookery", COOKERY_TRAIN)):
    sample = sorted(path.glob("*.txt"))[6].read_text(encoding="utf-8")
    print(f"--- {name} " + "-" * (66 - len(name)))
    print(textwrap.fill(" ".join(sample.split())[:520], 88), "…\n")

--- A / darwin --------------------------------------------------------
Sterility has been said to be the bane of horticulture; but on this view we owe
variability to the same cause which produces sterility; and variability is the source of
all the choicest productions of the garden. I may add, that as some organisms will breed
most freely under the most unnatural conditions (for instance, the rabbit and ferret
kept in hutches), showing that their reproductive system has not been thus affected; so
will some animals and plants withstand domestication or cultivation, and vary very
slight …

--- B / cookery -------------------------------------------------------
Celery Sauce. Take a large bunch of celery, cut it fine, and boil it till soft, in a
pint of water; thicken it with butter and flour, and season it with salt, pepper, and
mace. Bacon Dumplings. Cut slices of cooked bacon, and pepper them; roll out crust as
for apple dumplings; slice some potatoes very thin, and put them in the cru

## 3. A workspace

A workspace is the state machine that keeps a chain honest. It records **which model the next
stage adapts**, **which version of `p(h)` that stage anchors against**, and **the order the two
may be done in** — get any of those wrong and the run still completes, still reports a
plausible loss, and quietly measures something else.

`init` does not load the model; it records it. It copies the artifact in as `artifacts/v1.pt`,
so the workspace owns its own `p(h)` and later versions sit beside it. It picks up the bundled
recipe whose `model_id` is this model automatically.

The CLI equivalent of this cell is one line:

```bash
lfa init lfa_demo/workspace --model Qwen/Qwen3-0.6B --artifact qwen3-0.6b-gmm1543k-int8
```

In [7]:
from lfa import Recipe, Workspace

workspace = Workspace.init(WS, MODEL_ID, artifact=str(artifact_path), artifact_id=ARTIFACT_ID)
recipe = Recipe.load("qwen3-0.6b")

print(f"model            {workspace.state['model_id']}")
print(f"artifact         v{workspace.state['artifact_version']}  ({workspace.state['artifact_id']})")
print(f"stage            {workspace.state['stage']}")
print("\nthe operating point this recipe carries:")
for field in ("lora_rank", "lora_alpha", "lambda_qkv", "lambda_mlp", "mu", "anchor_end_ratio",
              "n_anchor_samples", "epochs", "learning_rate", "batch_size", "sequence_length",
              "val_fraction", "stage2_lambda_multiplier"):
    print(f"   {field:<26} {getattr(recipe, field)}")

notes = recipe.warnings(rank=recipe.lora_rank, artifact_id=ARTIFACT_ID)
print(f"\noff-calibration warnings: {notes if notes else 'none — this is the point λ was tuned at'}")

Local artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt recorded as the published 'qwen3-0.6b-gmm1543k-int8'


Artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace over Qwen/Qwen3-0.6B


model            Qwen/Qwen3-0.6B
artifact         v1  (qwen3-0.6b-gmm1543k-int8)
stage            0

the operating point this recipe carries:
   lora_rank                  32
   lora_alpha                 64
   lambda_qkv                 100000.0
   lambda_mlp                 100000.0
   mu                         0.05
   anchor_end_ratio           0.1
   n_anchor_samples           16
   epochs                     15
   learning_rate              0.0003
   batch_size                 6
   sequence_length            512
   val_fraction               0.1
   stage2_lambda_multiplier   3.0

off-calibration warnings: none — this is the point λ was tuned at


## 4. Stage 1 — adapt to domain A

One training run. Rank-32 LoRA over the frozen teacher, with three terms in the loss:

```
L  =  L_content                                    cross-entropy on Darwin
   +  λ · E_{h ~ p(h)} ‖f_student(h) − f_teacher(h)‖²    the function anchor
   +  μ · ‖W_student − W_teacher‖²                 the weight backstop
```

The middle term never touches the batch. On every step it draws 16 hidden states per site
**from the artifact**, runs the teacher's sub-module and the student's sub-module on them, and
penalizes the difference. That is the whole of the preservation signal, and it is why nothing
in this notebook ever needs the old data.

### On the number of epochs

The recipe's 15 epochs were tuned on a corpus of about 6.6 MB. These two are 1.45 MB together,
so roughly a quarter of the text, and on that much they over-train.

Count documents rather than bytes and you get a much larger-looking gap, because that corpus is
1,673 files of which 1,596 are short generated question-answer pairs making up only 14 % of its
text; the substance is 77 long documents. A file count would say the recipe saw nine times more
than this notebook. It saw about four and a half times more text. Bytes are the honest measure
here, and even they overstate it slightly, because those short files are dropped from most
epochs by the loader's re-chunking.

An 8-epoch pass
of exactly this configuration was run first, and the trainer's held-out perplexity bottomed at
epoch 5 (18.51, 17.62, 17.60, 17.31, **17.20**) and then climbed — 17.87, 18.99, 18.46.
`final_model` is the last epoch by design, so the fix is to re-run at the epoch it bottomed at,
which is what `EPOCHS = 5` above is. That choice was made on the **domain** axis of the
anchored run alone, before any control had been trained to compare it against, and the control
below is trained at the same 5 epochs. Section 8b returns to whether that is fair to it.

The CLI equivalent:

```bash
lfa train --workspace lfa_demo/workspace --corpus lfa_demo/data/darwin/train --epochs 5
```

In [8]:
t0 = time.time()
stage1 = workspace.train(DARWIN_TRAIN, epochs=EPOCHS, device=DEVICE)
stage1_seconds = time.time() - t0

print(f"\nstage {stage1['stage']}: {stage1['n_train_docs']} documents trained, "
      f"{stage1['n_val_docs']} held back for the curve, λ = {stage1['lambda_applied']:,.0f}, "
      f"{stage1_seconds/60:.1f} min")

loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


Stage 1: Qwen/Qwen3-0.6B on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/train (artifact v1, λ_qkv=100000.0, 5 epochs)


Layer weights (cosine, end_ratio=0.1): L0=0.0631, L27=0.0065


LR schedule: warmup(50) → cosine(275)


Starting training for 5 epochs


...

  Training state saved (epoch=5, step=286, checkpoint=latest_model)


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/final_model


  Training state saved (epoch=5, step=286, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/training_history.json



stage 1: 173 documents trained, 20 held back for the curve, λ = 100,000, 3.9 min


## 5. The control that makes the point

`evaluate` reads the stage on **both axes at once**, because either alone is meaningless: a run
that reports only the domain has not said what it gave up, and one that reports only WikiText-2
has not said whether it learned anything.

`compare_unanchored=True` adds a third column by training the stage **again** with
λ = μ = 0 — same corpus, same 5 epochs, same learning rate, same seed, same everything except
the anchor. That is a second full training run, and it is what says what the anchor bought.
Same corpus, same steps, so the difference is the anchor — with one caveat that section 8b
measures rather than assumes: 5 epochs is the *anchored* run's best dose, not the control's.

The domain axis here is Darwin's `heldout/` directory, passed explicitly. (`evaluate` logs
"a fit, not a held-out measurement" whenever a corpus is named, because a named corpus is not
split — but these documents were never in the training directory at all, so the number is
genuinely held out. Naming it is what makes it the *same* text as every later domain-A read.)

```bash
lfa evaluate --workspace lfa_demo/workspace --corpus lfa_demo/data/darwin/heldout \
             --compare-unanchored --n-windows 200
```

In [9]:
t0 = time.time()
scores1 = workspace.evaluate(corpus=DARWIN_HELD, compare_unanchored=True,
                             n_windows=N_WINDOWS, device=DEVICE)
control1_seconds = time.time() - t0
print("\n" + scores1["table"])

Evaluating stage 1 on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/heldout (64 chunks, trained on -- a fit, not a held-out measurement)


WikiText-2 perplexity 17.80 (loss 2.8792 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 30.12 (loss 3.4053 over 26147 tokens in 64 chunks)


WikiText-2 perplexity 16.40 (loss 2.7970 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 17.59 (loss 2.8676 over 26147 tokens in 64 chunks)


...

Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1_unanchored/training_history.json


This run trained past its own optimum: held-out perplexity was lowest at epoch 2 (15.018) and ended at 19.221, epoch 5. The shipped recipe's epoch count was tuned on a corpus of about 1,700 documents; a smaller one reaches its minimum in far fewer epochs. `final_model` is the LAST epoch -- no best checkpoint is kept, by design (docs/recipes.md says why) -- so to ship the better model, re-run with --epochs 2, which lays the whole learning-rate schedule over that many epochs rather than truncating this one.


WikiText-2 perplexity 37.18 (loss 3.6157 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 19.78 (loss 2.9848 over 26147 tokens in 64 chunks)



| metric               | before | after |     Δ% | unanchored |      Δ% |
| -------------------- | -----: | ----: | -----: | ---------: | ------: |
| general (WikiText-2) |  17.80 | 16.40 |  -7.9% |      37.18 | +108.9% |
| domain               |  30.12 | 17.59 | -41.6% |      19.78 |  -34.3% |


In [10]:
before, after, unanchored = scores1["before"], scores1["after"], scores1["unanchored"]
print(f"""
Read it as two questions.

  Did it learn Darwin?   {before['domain']:.2f} -> {after['domain']:.2f}
                         ({(after['domain']/before['domain'] - 1) * 100:+.1f}% held-out domain perplexity)

  What did that cost?    WikiText-2 {before['general']:.2f} -> {after['general']:.2f}
                         ({(after['general']/before['general'] - 1) * 100:+.1f}%)

  Without the anchor:    Darwin      {before['domain']:.2f} -> {unanchored['domain']:.2f}   ({(unanchored['domain']/before['domain'] - 1) * 100:+.1f}%)
                         WikiText-2  {before['general']:.2f} -> {unanchored['general']:.2f}   ({(unanchored['general']/before['general'] - 1) * 100:+.1f}%)
""")


Read it as two questions.

  Did it learn Darwin?   30.12 -> 17.59
                         (-41.6% held-out domain perplexity)

  What did that cost?    WikiText-2 17.80 -> 16.40
                         (-7.9%)

  Without the anchor:    Darwin      30.12 -> 19.78   (-34.3%)
                         WikiText-2  17.80 -> 37.18   (+108.9%)



## 6. `extend` — folding A in, so that B never has to see it

A second domain is not a second training run. Stage 2 has to adapt the model that already
carries A, and it has to anchor against a `p(h)` that *describes* that model — otherwise it
preserves a function the model no longer computes. `extend` does both, and the workspace
refuses to train a different corpus until it has:

* **Merge.** Stage 1's adapter is merged into the model it trained over, giving
  `models/stage1_fused` — a plain checkpoint. That is what stage 2 adapts.
* **Extend `p(h)`.** Domain A is run through the *fused* model, its activations are fitted as a
  small mixture **in the base artifact's own PCA basis**, and the two mixtures are merged as a
  sample-weighted union:

  ```
  p(h) = (1 − a) · p_base(h) + a · p_domain(h),      a = n_domain / (n_base + n_domain)
  ```

The union is exact — it is the distribution of "draw from the base pool with probability
1 − a, else from the domain pool" — so nothing is refitted, and the merged artifact is a
sufficient statistic for the round after that as well. Each round reads only the **new** domain
and adds a fixed number of components per site, so the cost per round is flat in the number of
rounds. Replay has to rehearse every prior corpus at every stage; this does not.

The output is `artifacts/v2.pt`, and it is still a set of per-site tensors. **No Darwin text
is stored anywhere in the workspace from here on** — the cell after this one checks that.

`--need` is the memory bill: the collection keeps one `need x width` float32 buffer
per site, on the host, for all 84 sites at once — about 9 GB of RAM at the 20,000
used here, and about 18 GB at the 40,000 default.

```bash
lfa extend --workspace lfa_demo/workspace --need 20000 --k-domain 4
```

In [11]:
t0 = time.time()
artifact_v2 = workspace.extend(need=EXTEND_NEED, k_domain=EXTEND_K, device=DEVICE)
extend_seconds = time.time() - t0
print(f"\n{artifact_v2}  ({artifact_v2.stat().st_size / 1e6:.1f} MB, {extend_seconds/60:.1f} min)")
print(f"current model    {workspace.state['current_model']}")
print(f"current artifact v{workspace.state['artifact_version']}")

Fusing adapter /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/final_model into Qwen/Qwen3-0.6B


Fused model and tokenizer written to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Collected 20000 activations at each of 84 sites from 52 chunks


Fitting the domain mixture: site 1/84 (0_pre_o)


...

Fitting the domain mixture: site 80/84 (26_pre_mlp)


Fitting the domain mixture: site 84/84 (28_pre_lm_head)


Extended 84 sites with K 4-4 new components from 20000 activations (domain weight share 0.0128); wrote version 2


Stage 1 folded in: model /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused, artifact v2



/tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v2.pt  (113.3 MB, 1.1 min)
current model    /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused
current artifact v2


In [12]:
v1, v2 = load_artifact(artifact_path), load_artifact(artifact_v2)
site = "12_pre_mlp"
basis_shift = (v1[site]["pca_components"].float()
               - v2[site]["pca_components"].float()).abs().max().item()

print(f"{site}: mixture components  v1 {v1[site]['gmm_n_components']}"
      f"  ->  v2 {v2[site]['gmm_n_components']}   (+{EXTEND_K} for domain A)")
print(f"          basis: largest entry-wise change {basis_shift:.2e} — the domain mixture is")
print(f"                 fitted IN the base basis, so this is int8 round-trip noise, not a refit")
print(f"          value types: {sorted({type(x).__name__ for x in v2[site].values()})}")


# The whole point of the method, stated as a filesystem fact: search every byte the workspace
# holds — adapters, optimizer state, the fused model, both artifacts — for a phrase the domain-A
# corpus is full of.
def contains(path, needle, chunk=1 << 22):
    with path.open("rb") as handle:
        tail = b""
        while (block := handle.read(chunk)):
            if needle in tail + block:
                return True
            tail = block[-len(needle):]
    return False


NEEDLE = b"natural selection"
files = [p for p in WS.rglob("*") if p.is_file()]
carriers = [p.relative_to(WS) for p in files if contains(p, NEEDLE)]
occurrences = sum(doc.read_bytes().count(NEEDLE) for doc in DARWIN_TRAIN.glob("*.txt"))

print(f"\n{NEEDLE.decode()!r} appears {occurrences} times in the domain-A training corpus.")
print(f"Files in the workspace ({len(files)} files, "
      f"{sum(p.stat().st_size for p in files) / 1e9:.2f} GB) that contain it: {len(carriers)}")

12_pre_mlp: mixture components  v1 32  ->  v2 36   (+4 for domain A)
          basis: largest entry-wise change 0.00e+00 — the domain mixture is
                 fitted IN the base basis, so this is int8 round-trip noise, not a refit
          value types: ['Tensor', 'float', 'int', 'str']



'natural selection' appears 186 times in the domain-A training corpus.
Files in the workspace (32 files, 2.08 GB) that contain it: 0


## 7. Stage 2 — adapt to domain B

Same command as stage 1. Three things are different, and the workspace supplies all three:

* it adapts `models/stage1_fused`, not the base model;
* it anchors against `artifacts/v2.pt`, which now describes *that* model on *both* domains;
* λ is multiplied by the recipe's `stage2_lambda_multiplier` (3.0), because a later stage is
  holding more than the base model's functions in place. That multiplier is a **level, not a
  compounding factor** — stage 3 would anchor at the same multiple, not its square — and the
  shipped 3.0 is a starting default rather than a calibrated constant.

Nothing from Darwin is read. There is no buffer, no rehearsal set, no replay mixture.

```bash
lfa train --workspace lfa_demo/workspace --corpus lfa_demo/data/cookery/train --epochs 5
```

In [13]:
t0 = time.time()
stage2 = workspace.train(COOKERY_TRAIN, epochs=EPOCHS, device=DEVICE)
stage2_seconds = time.time() - t0

print(f"\nstage {stage2['stage']}: {stage2['n_train_docs']} documents, "
      f"λ = {stage2['lambda_applied']:,.0f} "
      f"(= {recipe.lambda_qkv:,.0f} × {recipe.stage2_lambda_multiplier}), "
      f"anchored against artifact v{stage2['artifact_version']}, {stage2_seconds/60:.1f} min")
print(f"          starting model: {stage2['base_model']}")

loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


Stage 2: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused on /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/train (artifact v2, λ_qkv=300000.0, 5 epochs)


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Layer weights (cosine, end_ratio=0.1): L0=0.0631, L27=0.0065


LR schedule: warmup(50) → cosine(120)


...

  Training state saved (epoch=5, step=156, checkpoint=latest_model)


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/final_model


  Training state saved (epoch=5, step=156, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/training_history.json



stage 2: 94 documents, λ = 300,000 (= 100,000 × 3.0), anchored against artifact v2, 2.1 min
          starting model: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


## 8. The contrast again, and the question the whole notebook is about

First the same two-axis read as before, with its own unanchored control: stage 2 re-run from
the same starting model with λ = μ = 0.

In [14]:
t0 = time.time()
scores2 = workspace.evaluate(corpus=COOKERY_HELD, compare_unanchored=True,
                             n_windows=N_WINDOWS, device=DEVICE)
control2_seconds = time.time() - t0
print("\n" + scores2["table"])

The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Evaluating stage 2 on /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/heldout (33 chunks, trained on -- a fit, not a held-out measurement)


WikiText-2 perplexity 16.39 (loss 2.7967 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 23.80 (loss 3.1697 over 14949 tokens in 33 chunks)


WikiText-2 perplexity 16.23 (loss 2.7867 over 102400 tokens, window 2048 / stride 512)


...

Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2_unanchored/training_history.json


This run trained past its own optimum: held-out perplexity was lowest at epoch 2 (12.138) and ended at 13.788, epoch 5. The shipped recipe's epoch count was tuned on a corpus of about 1,700 documents; a smaller one reaches its minimum in far fewer epochs. `final_model` is the LAST epoch -- no best checkpoint is kept, by design (docs/recipes.md says why) -- so to ship the better model, re-run with --epochs 2, which lays the whole learning-rate schedule over that many epochs rather than truncating this one.


WikiText-2 perplexity 27.93 (loss 3.3297 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 14.82 (loss 2.6962 over 14949 tokens in 33 chunks)



| metric               | before | after |     Δ% | unanchored |     Δ% |
| -------------------- | -----: | ----: | -----: | ---------: | -----: |
| general (WikiText-2) |  16.39 | 16.23 |  -1.0% |      27.93 | +70.4% |
| domain               |  23.80 | 15.10 | -36.5% |      14.82 | -37.7% |


That table's "before" column is the model **after domain A**, so it reads stage 2 in isolation.
It does not yet answer the question the method exists for, which is about A.

So: five models, three axes, one table. The general axis is WikiText-2, which no stage ever
trained on. The two domain axes are the `heldout/` documents, which no stage ever trained on
either. Every row is the same three measurements on the same three texts.

In [15]:
import gc
from peft import PeftModel
from lfa.corpus import load_corpus
from lfa.evaluate import domain_perplexity, wikitext2_perplexity
from lfa.models import load_teacher, load_tokenizer

tokenizer = load_tokenizer(MODEL_ID)


def held_out(path):
    corpus, _ = load_corpus(path, tokenizer, max_length=recipe.sequence_length,
                            val_fraction=0.0, seed=recipe.seed,
                            keep_short_whole=recipe.keep_short_whole)
    return corpus


DOMAINS = {"A · Darwin": held_out(DARWIN_HELD), "B · cookery": held_out(COOKERY_HELD)}
STAGE1_FUSED = WS / "models" / "stage1_fused"

MODELS = {
    "base Qwen3-0.6B":            (MODEL_ID, None),
    "after A — anchored":         (MODEL_ID, WS / "runs" / "stage1" / "final_model"),
    "after A — NO anchor":        (MODEL_ID, WS / "runs" / "stage1_unanchored" / "final_model"),
    "after A+B — anchored":       (STAGE1_FUSED, WS / "runs" / "stage2" / "final_model"),
    "after A+B — NO anchor on B": (STAGE1_FUSED, WS / "runs" / "stage2_unanchored" / "final_model"),
}


def score(base, adapter):
    model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter))
    try:
        row = {"WikiText-2": wikitext2_perplexity(model, tokenizer, n_windows=N_WINDOWS,
                                                  device=DEVICE)}
        row.update({name: domain_perplexity(model, tokenizer, corpus, device=DEVICE)
                    for name, corpus in DOMAINS.items()})
        return row
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


logging.getLogger("lfa.evaluate").setLevel(logging.WARNING)   # the table is the output here
t0 = time.time()
table = {label: score(*spec) for label, spec in MODELS.items()}
final_table_seconds = time.time() - t0

In [16]:
columns = ["WikiText-2", "A · Darwin", "B · cookery"]
width = max(len(label) for label in table)
base_row = table["base Qwen3-0.6B"]

header = f"| {'model':<{width}} | " + " | ".join(f"{c:>18}" for c in columns) + " |"
print(header)
print(f"| {'-' * width} | " + " | ".join("-" * 18 for _ in columns) + " |")
for label, row in table.items():
    cells = []
    for column in columns:
        delta = (row[column] / base_row[column] - 1) * 100
        cells.append(f"{row[column]:8.2f}" + ("         " if label.startswith("base")
                                              else f" ({delta:+5.1f}%)"))
    print(f"| {label:<{width}} | " + " | ".join(f"{c:>18}" for c in cells) + " |")
print("\nperplexity, lower is better; % is against the base model on the same text")

| model                      |         WikiText-2 |         A · Darwin |        B · cookery |
| -------------------------- | ------------------ | ------------------ | ------------------ |
| base Qwen3-0.6B            |     17.80          |     30.12          |     26.51          |
| after A — anchored         |     16.40 ( -7.9%) |     17.59 (-41.6%) |     23.80 (-10.2%) |
| after A — NO anchor        |    37.18 (+108.9%) |     19.78 (-34.3%) |     45.25 (+70.7%) |
| after A+B — anchored       |     16.23 ( -8.8%) |     19.10 (-36.6%) |     15.10 (-43.0%) |
| after A+B — NO anchor on B |     27.93 (+56.9%) |     38.78 (+28.7%) |     14.82 (-44.1%) |

perplexity, lower is better; % is against the base model on the same text


### How to read that table

Four comparisons, in the order they answer the notebook's question.

1. **Row 2 against row 1** — did the anchored stage learn Darwin, and what did WikiText-2 do?
2. **Row 3 against row 2** — the control. Same corpus, same steps, anchor off. The unanchored
   run reaches the domain *harder*; the general column is where it pays for it. Read that gap
   as an **upper bound** on what the anchor bought, not as the anchor alone: both controls are
   trained at the anchored run's number of epochs, which is past their own optimum. Section 8b
   re-runs them at their own dose and separates the two.
3. **Row 4's `A · Darwin` cell against row 2's** — the chain question. Stage 2 trained on
   nothing but recipes, and read no Darwin text at all. How much of A survived?
4. **Row 5's `A · Darwin` cell against row 4's** — the same stage without the anchor. Both start
   from the same model that already knows A; only one of them is holding it. The same upper-bound
   caveat applies, and section 8b measures this one too.

A caution that belongs beside the numbers: **this is one run per cell at one seed**, on corpora
an order of magnitude smaller than the operating point was tuned on, at a third of its epochs.
Read the *direction and size* of each contrast, not the digits. Two more warnings from
`docs/concepts.md` apply directly here: a general perplexity *below* the base model's is not by
itself evidence that anything was preserved, and over-anchoring makes the general axis look its
best while the domain quietly collapses — which is why nothing in this notebook is read on one
axis alone.

In [17]:
a_after_a = table["after A — anchored"]["A · Darwin"]
a_after_ab = table["after A+B — anchored"]["A · Darwin"]
a_after_ab_free = table["after A+B — NO anchor on B"]["A · Darwin"]
g_anchored = table["after A — anchored"]["WikiText-2"]
g_free = table["after A — NO anchor"]["WikiText-2"]

print(f"""
1. stage 1, anchored     Darwin {base_row['A · Darwin']:.2f} -> {a_after_a:.2f}    WikiText-2 {base_row['WikiText-2']:.2f} -> {g_anchored:.2f}
2. stage 1, no anchor    Darwin {base_row['A · Darwin']:.2f} -> {table['after A — NO anchor']['A · Darwin']:.2f}    WikiText-2 {base_row['WikiText-2']:.2f} -> {g_free:.2f}
   what the anchor bought on the general axis: {(g_free / g_anchored - 1) * 100:+.1f}% worse without it

3. domain A across stage 2, anchored:     {a_after_a:.2f} -> {a_after_ab:.2f}   ({(a_after_ab / a_after_a - 1) * 100:+.1f}%)
4. domain A across stage 2, anchor off:   {a_after_a:.2f} -> {a_after_ab_free:.2f}   ({(a_after_ab_free / a_after_a - 1) * 100:+.1f}%)
   stage 2 read 0 words of Darwin in either case.
""")


1. stage 1, anchored     Darwin 30.12 -> 17.59    WikiText-2 17.80 -> 16.40
2. stage 1, no anchor    Darwin 30.12 -> 19.78    WikiText-2 17.80 -> 37.18
   what the anchor bought on the general axis: +126.8% worse without it

3. domain A across stage 2, anchored:     17.59 -> 19.10   (+8.6%)
4. domain A across stage 2, anchor off:   17.59 -> 38.78   (+120.4%)
   stage 2 read 0 words of Darwin in either case.



## 8b. Is the control fair? Both stages, at their own dose

There is a real objection to everything above, and it is worth measuring rather than waving at.

`EPOCHS = 5` was chosen from the **anchored** stage-1 run's held-out curve. Both unanchored
controls are trained at that same 5, because "the same run without the anchor" is what a control
is — but 5 is not *their* best dose. Without the anchor the model reaches the domain much faster
and then over-fits, and the trainer said so itself at the end of both control runs ("this run
trained past its own optimum"). So part of every gap above is the anchor and part of it is three
extra epochs of unregularized training. Both curves are printed below, so the size of that
effect is visible rather than asserted.

The honest version of the question is: **at each run's own best dose, what is left?** Two more
runs answer it — each control again at 2 epochs, with its own complete learning-rate schedule,
each in a workspace of its own so the chain built above is untouched.

In [18]:
import dataclasses
import json as _json


def curve(run_dir):
    return [(e["epoch"], e["val_perplexity"])
            for e in _json.loads((Path(run_dir) / "training_history.json").read_text())
            if "val_perplexity" in e]


print("held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set")
for stage, runs in (("stage 1 (darwin) ", ("stage1", "stage1_unanchored")),
                    ("stage 2 (cookery)", ("stage2", "stage2_unanchored"))):
    print(f"  {stage}")
    for label, run in zip(("anchored ", "no anchor"), runs):
        print(f"     {label}  " + "  ".join(f"e{n}:{v:6.2f}" for n, v in curve(WS / "runs" / run)))

held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set
  stage 1 (darwin) 
     anchored   e1: 18.77  e2: 17.76  e3: 17.54  e4: 17.18  e5: 17.16
     no anchor  e1: 16.34  e2: 15.02  e3: 15.98  e4: 17.57  e5: 19.22
  stage 2 (cookery)
     anchored   e1: 16.01  e2: 15.01  e3: 14.69  e4: 14.46  e5: 14.23
     no anchor  e1: 13.75  e2: 12.14  e3: 12.32  e4: 13.16  e5: 13.79


In [19]:
control = dataclasses.replace(recipe, lambda_qkv=0.0, lambda_mlp=0.0, mu=0.0)
t0 = time.time()

# Stage 1's control at its own dose: the base model, 2 epochs, no anchor.
FAIR1 = DEMO / "workspace_dose_a"
fair1 = Workspace.init(FAIR1, MODEL_ID, artifact=str(artifact_path), artifact_id=ARTIFACT_ID)
fair1.train(DARWIN_TRAIN, recipe=control, epochs=2, device=DEVICE)

# Stage 2's control at its own dose: the SAME starting model stage 2 had (the anchored stage-1
# fused checkpoint) and the same corpus, 2 epochs, no anchor. A workspace of its own, so the
# chain above keeps pointing at its real stage 2.
FAIR2 = DEMO / "workspace_dose_b"
fair2 = Workspace.init(FAIR2, str(STAGE1_FUSED), artifact=str(artifact_v2),
                       artifact_id=ARTIFACT_ID, recipe="qwen3-0.6b")
fair2.train(COOKERY_TRAIN, recipe=control, epochs=2, device=DEVICE)

dose_seconds = time.time() - t0
print(f"\n{dose_seconds / 60:.1f} min")
print("  stage-1 control, 2 epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                  for n, v in curve(FAIR1 / "runs" / "stage1")))
print("  stage-2 control, 2 epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                  for n, v in curve(FAIR2 / "runs" / "stage1")))

Local artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt recorded as the published 'qwen3-0.6b-gmm1543k-int8'


Artifact /tmp/lfa-nb-run/cwd/lfa_demo/artifacts/qwen3-0.6b-gmm1543k-int8.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a over Qwen/Qwen3-0.6B


loader frame: short documents are kept whole in every epoch; a run under keep_short_whole=False sees them in fewer epochs, and perplexities are not comparable across the two


...

  Held-out: loss=2.4942 perplexity=12.112 over 10503 tokens


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/final_model


  Training state saved (epoch=2, step=67, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/training_history.json



2.6 min
  stage-1 control, 2 epochs: e1: 16.45  e2: 15.28
  stage-2 control, 2 epochs: e1: 13.76  e2: 12.11


In [20]:
dose_a = score(MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model")
dose_b = score(STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model")


def block(title, rows, columns):
    width = max(len(label) for label, _ in rows)
    print(f"\n{title}")
    print(f"| {'':<{width}} | " + " | ".join(f"{c:>12}" for c in columns) + " |")
    print(f"| {'-' * width} | " + " | ".join("-" * 12 for _ in columns) + " |")
    for label, row in rows:
        print(f"| {label:<{width}} | " + " | ".join(f"{row[c]:12.2f}" for c in columns) + " |")


block("stage 1 — adapting the BASE model to Darwin",
      [("base model", base_row),
       ("anchored, 5 epochs", table["after A — anchored"]),
       ("no anchor, 5 epochs", table["after A — NO anchor"]),
       ("no anchor, 2 epochs", dose_a)],
      ["WikiText-2", "A · Darwin"])

block("stage 2 — adapting the same after-A model to the cookbook",
      [("model it started from", table["after A — anchored"]),
       ("anchored, 5 epochs", table["after A+B — anchored"]),
       ("no anchor, 5 epochs", table["after A+B — NO anchor on B"]),
       ("no anchor, 2 epochs", dose_b)],
      ["WikiText-2", "A · Darwin", "B · cookery"])


stage 1 — adapting the BASE model to Darwin
|                     |   WikiText-2 |   A · Darwin |
| ------------------- | ------------ | ------------ |
| base model          |        17.80 |        30.12 |
| anchored, 5 epochs  |        16.40 |        17.59 |
| no anchor, 5 epochs |        37.18 |        19.78 |
| no anchor, 2 epochs |        19.14 |        15.63 |

stage 2 — adapting the same after-A model to the cookbook
|                       |   WikiText-2 |   A · Darwin |  B · cookery |
| --------------------- | ------------ | ------------ | ------------ |
| model it started from |        16.40 |        17.59 |        23.80 |
| anchored, 5 epochs    |        16.23 |        19.10 |        15.10 |
| no anchor, 5 epochs   |        27.93 |        38.78 |        14.82 |
| no anchor, 2 epochs   |        18.02 |        23.49 |        12.95 |


Read those on **all** the columns at once, which is the whole discipline of the method. The
question is never "which run has the better WikiText-2"; it is "at comparable fit on the domain
being learned, which run kept more of everything else". A control that is worse everywhere has
been beaten outright. A control that is *better* on the new domain and worse elsewhere has
bought the domain by spending what it already had — which is the trade the anchor exists to
change, not a trade it claims to abolish.

Two things this section is honest about, whichever way the numbers fell:

* Some of the gap in section 5 and section 8 is dose, not anchor. Reporting only the 5-epoch
  controls would have overstated the anchor by the difference between the two control rows here.
* A 2-epoch run is not simply a truncated 5-epoch run: the learning-rate schedule is laid over
  whatever number of epochs you ask for, so these are complete runs with their own warmup and
  decay. That is the right comparison, and it is also a *second* difference from the 5-epoch
  control, not a clean single-variable change.

And, still: one seed, one run per row, 193 documents.

## 9. Export: a plain checkpoint

`fuse` merges the last stage's adapter into the model it trained over and writes an ordinary
Hugging Face checkpoint — no PEFT wrapper, no `lfa` at load time, nothing to configure. It
loads with `AutoModelForCausalLM.from_pretrained` like any other model.

```bash
lfa fuse --workspace lfa_demo/workspace
```

In [21]:
exported = workspace.fuse()
print(exported)
print(sorted(p.name for p in exported.iterdir()))

Fusing adapter /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/final_model into /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Fused model and tokenizer written to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export


/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export
['added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer

merged = AutoModelForCausalLM.from_pretrained(str(exported), dtype=torch.bfloat16).to(DEVICE)
merged_tokenizer = AutoTokenizer.from_pretrained(str(exported))

PROMPTS = [
    "The several difficulties which have been discussed, namely, that",   # domain A
    "To make a plain beef soup, take",                                    # domain B
]
for prompt in PROMPTS:
    ids = merged_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = merged.generate(**ids, max_new_tokens=60, do_sample=False,
                              pad_token_id=merged_tokenizer.eos_token_id)
    print("· " + textwrap.fill(merged_tokenizer.decode(out[0], skip_special_tokens=True),
                               88, subsequent_indent="  ") + "\n")

del merged
gc.collect()
torch.cuda.empty_cache()

The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


· The several difficulties which have been discussed, namely, that of the necessity of a
  new system of classification, and of the influence of the geographical distribution of
  the species, and of the influence of the climate, and of the soil, and of the animals
  and plants which inhabit the country, and of the relations of the species to



· To make a plain beef soup, take a pound of beef, cut it up, and put it in a pot with a
  pint of water, and a pound of onions, a pound of pepper, a pound of salt, and a pound
  of flour; put in a pint of water, and boil it for half an hour



Those two continuations are **qualitative only** — a greedy decode from one small model, not a
measurement, and nothing in this package scores generated text. They are here because it is
worth seeing that one set of weights answers in both registers.

## 10. The whole chain, and what it cost

In [23]:
import json

for entry in json.load((WS / "history.json").open()):
    perplexity = entry.get("perplexity") or {}
    after = (perplexity.get("after") or {})
    print(f"stage {entry['stage']}  {Path(entry['corpus']).parent.name:<9} "
          f"λ={entry['lambda_applied']:>9,.0f}  artifact v{entry['artifact_version']}  "
          f"{entry['n_train_docs']:>3} docs  "
          f"domain {after.get('domain', float('nan')):.2f}  "
          f"general {after.get('general', float('nan')):.2f}")

print(f"""
wall clock
  stage 1 (anchored)                {stage1_seconds / 60:5.1f} min
  stage 1 evaluate + control        {control1_seconds / 60:5.1f} min
  extend (fuse + fit + merge p(h))  {extend_seconds / 60:5.1f} min
  stage 2 (anchored)                {stage2_seconds / 60:5.1f} min
  stage 2 evaluate + control        {control2_seconds / 60:5.1f} min
  the five-model table              {final_table_seconds / 60:5.1f} min
  the two dose-matched controls     {dose_seconds / 60:5.1f} min
  ------------------------------------------
  notebook, end to end              {(time.time() - NOTEBOOK_STARTED) / 60:5.1f} min
""")

stage 1  darwin    λ=  100,000  artifact v1  173 docs  domain 17.59  general 16.40
stage 2  cookery   λ=  300,000  artifact v2   94 docs  domain 15.10  general 16.23

wall clock
  stage 1 (anchored)                  3.9 min
  stage 1 evaluate + control          4.7 min
  extend (fuse + fit + merge p(h))    1.1 min
  stage 2 (anchored)                  2.1 min
  stage 2 evaluate + control          3.1 min
  the five-model table                2.4 min
  the two dose-matched controls       2.6 min
  ------------------------------------------
  notebook, end to end               21.1 min



## What this showed, and what it did not

**Showed.** Two domains adapted in sequence on one 0.6B model, with the second stage reading
none of the first domain's text and the workspace storing none of it. Both stages measured on
two axes against a control that differs only in λ and μ, and both of those controls
measured again at their own best dose, because a control run past its optimum flatters the
thing it is controlling for.

**Did not show.** Anything statistical. One seed, one run per cell, corpora roughly a tenth of
what the operating point was tuned on, a third of its epochs, and no repetition — so a
difference of a few percent in any cell is not distinguishable from run-to-run variation here.
It also did not reproduce any published number, and does not try to: the paper's own headline
is its measurement on its corpus and its instruments.

**If you want to push it further**, the honest next steps are the boring ones: run each cell at
a second seed, put the shipped 15 epochs on a corpus the size they were tuned for, and re-tune
λ if you change the rank or the artifact — it is coupled to both, and a ported λ measures the
port rather than the method.

### Where to go next

| | |
|---|---|
| `docs/concepts.md` | what the anchor does, what λ and μ are, how a run is read |
| `docs/recipes.md` | the shipped operating point field by field, and its couplings |
| `docs/multi-domain-chains.md` | a third domain, `lfa chain`, and what the paper found after one |
| `docs/faq.md` | GPU memory, full weights, reading the general axis |
| `examples/quickstart.py` | the single-domain flow as a script |